# 03 — Merchant Segmentation

**Goal:** Segment the ~3,000 Olist sellers into four actionable tiers using RFM (Recency, Frequency, Monetary) features and K-means clustering. Each segment receives a labeled profile and a product recommendation stub.

**Approach:**
1. Build RFM features; log-transform skewed distributions; standardize.
2. Validate k with elbow method and silhouette analysis.
3. Fit K-means (k=4). Map clusters to segment names based on RFM centroids.
4. Compute segment profiles: size, GMV share, delivery reliability, review scores, top categories, declining trajectory rate.
5. Visualize with a 2×2 Plotly grid.
6. Save segmented table; write segment summary stubs.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from pathlib import Path

RAW       = Path('../data/raw')
PROCESSED = Path('../data/processed')

# Consistent palette across the project
SEGMENT_COLORS = {
    'Champions':   '#2ecc71',   # green  — healthy, high-value
    'Rising Stars':'#3498db',   # blue   — growing
    'At Risk':     '#f39c12',   # amber  — needs attention
    'Dormant':     '#e74c3c',   # red    — inactive
}
SEGMENT_ORDER = ['Champions', 'Rising Stars', 'At Risk', 'Dormant']

print('Libraries loaded.')

Libraries loaded.


---
## 1. Load Data

In [2]:
mf = pd.read_csv(
    PROCESSED / 'merchant_features.csv',
    parse_dates=['first_order_date', 'last_order_date']
)
mt = pd.read_csv(PROCESSED / 'merchant_trajectory.csv')[['seller_id', 'trajectory', 'is_declining']]

print(f'merchant_features:  {mf.shape[0]:,} sellers, {mf.shape[1]} columns')
print(f'merchant_trajectory: {mt.shape[0]:,} sellers (active in both time periods)')
print(f'seller_id overlap:   {len(set(mf.seller_id) & set(mt.seller_id)):,}')

merchant_features:  2,960 sellers, 25 columns
merchant_trajectory: 836 sellers (active in both time periods)
seller_id overlap:   836


---
## 2. Build and Standardize RFM Features

- **Recency:** `recency_days` — days since last order (lower = more recent = better)
- **Frequency:** `order_count` — total orders across the dataset
- **Monetary:** `total_gmv` — cumulative price of all items sold (excl. freight)

Both frequency and GMV are heavily right-skewed (a small number of sellers dominate). Log-transforming before standardizing prevents the top sellers from collapsing all other merchants into a single cluster.

In [3]:
rfm = mf[['seller_id', 'recency_days', 'order_count', 'total_gmv']].dropna().copy()

# Log-transform the skewed dimensions (log1p avoids log(0))
rfm['log_frequency'] = np.log1p(rfm['order_count'])
rfm['log_monetary']  = np.log1p(rfm['total_gmv'])

print('Pre-transform skewness:')
print(f'  order_count skew: {rfm["order_count"].skew():.2f}')
print(f'  total_gmv skew:   {rfm["total_gmv"].skew():.2f}')
print(f'\nPost-transform skewness:')
print(f'  log_frequency skew: {rfm["log_frequency"].skew():.2f}')
print(f'  log_monetary skew:  {rfm["log_monetary"].skew():.2f}')

RFM_FEATURES = ['recency_days', 'log_frequency', 'log_monetary']
scaler = StandardScaler()
X = scaler.fit_transform(rfm[RFM_FEATURES])

print(f'\nFeature matrix shape: {X.shape}')
print(f'Standardized means (should be ~0): {X.mean(axis=0).round(10)}')
print(f'Standardized stds  (should be 1):  {X.std(axis=0).round(4)}')

Pre-transform skewness:
  order_count skew: 9.45
  total_gmv skew:   8.87

Post-transform skewness:
  log_frequency skew: 0.83
  log_monetary skew:  0.10

Feature matrix shape: (2960, 3)
Standardized means (should be ~0): [ 0.  0. -0.]
Standardized stds  (should be 1):  [1. 1. 1.]


---
## 3. Select Optimal k — Elbow Method and Silhouette Analysis

In [4]:
K_RANGE = range(2, 9)
inertias    = []
sil_scores  = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, labels))
    print(f'k={k}  inertia={km.inertia_:>8.1f}  silhouette={sil_scores[-1]:.4f}')

best_sil_k = list(K_RANGE)[np.argmax(sil_scores)]
print(f'\nBest silhouette score at k={best_sil_k} (score={max(sil_scores):.4f})')

k=2  inertia=  4721.2  silhouette=0.4014


k=3  inertia=  2967.7  silhouette=0.4177
k=4  inertia=  2236.6  silhouette=0.3716


k=5  inertia=  1878.3  silhouette=0.3759


k=6  inertia=  1599.8  silhouette=0.3351


k=7  inertia=  1400.9  silhouette=0.3363


k=8  inertia=  1265.4  silhouette=0.3077

Best silhouette score at k=3 (score=0.4177)


In [5]:
fig_val = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Elbow Method — Inertia by k<br><sup>Look for the "elbow" where improvement flattens</sup>',
        'Silhouette Score by k<br><sup>Higher = more distinct clusters; peak indicates best k</sup>'
    ),
    horizontal_spacing=0.12
)

ks = list(K_RANGE)

fig_val.add_trace(
    go.Scatter(
        x=ks, y=inertias, mode='lines+markers',
        marker=dict(size=8, color='#3498db'),
        line=dict(color='#3498db', width=2),
        hovertemplate='k=%{x}<br>Inertia=%{y:.1f}<extra></extra>'
    ),
    row=1, col=1
)

fig_val.add_trace(
    go.Scatter(
        x=ks, y=sil_scores, mode='lines+markers',
        marker=dict(size=8, color='#2ecc71'),
        line=dict(color='#2ecc71', width=2),
        hovertemplate='k=%{x}<br>Silhouette=%{y:.4f}<extra></extra>'
    ),
    row=1, col=2
)

# Mark k=4 on both charts
fig_val.add_vline(x=4, line_dash='dash', line_color='#e74c3c', line_width=1.5, row=1, col=1,
                  annotation_text='k=4 (selected)', annotation_position='top right')
fig_val.add_vline(x=4, line_dash='dash', line_color='#e74c3c', line_width=1.5, row=1, col=2,
                  annotation_text='k=4 (selected)', annotation_position='top right')

fig_val.update_xaxes(title_text='Number of Clusters (k)', dtick=1)
fig_val.update_yaxes(title_text='Inertia (within-cluster sum of squares)', row=1, col=1, gridcolor='#eee')
fig_val.update_yaxes(title_text='Silhouette Score', row=1, col=2, gridcolor='#eee')

fig_val.update_layout(
    title=dict(text='K-Means Cluster Validation: Elbow Method and Silhouette Analysis', font_size=16),
    plot_bgcolor='white', paper_bgcolor='white',
    showlegend=False, height=420, margin=dict(t=100)
)

fig_val.show()
print(f"Takeaway: The elbow bends at k=3–4; silhouette peaks at k={best_sil_k} but remains "
      f"reasonable at k=4 (score={sil_scores[2]:.3f}). k=4 is selected to produce four "
      f"actionable, interpretable segments aligned with product goals.")

Takeaway: The elbow bends at k=3–4; silhouette peaks at k=3 but remains reasonable at k=4 (score=0.372). k=4 is selected to produce four actionable, interpretable segments aligned with product goals.


**Takeaway:** The elbow bends between k=3 and k=4, and the silhouette score peaks at k=3 (0.418) before declining. I chose k=4 because it produces four segments that map cleanly to actionable product tiers (Champions, Rising Stars, At Risk, and Dormant) while still maintaining a silhouette score of 0.372. The small drop in cluster compactness felt like a reasonable trade-off for the added interpretability.

---
## 4. Fit K-Means (k=4) and Assign Segment Labels

In [6]:
K_SELECTED = 4
km = KMeans(n_clusters=K_SELECTED, random_state=42, n_init=10)
rfm['cluster'] = km.fit_predict(X)

# Inspect centroids in original scale to assign names
centroids = pd.DataFrame(
    scaler.inverse_transform(km.cluster_centers_),
    columns=RFM_FEATURES
)
centroids['log_frequency_exp'] = np.expm1(centroids['log_frequency']).round(0)
centroids['log_monetary_exp']  = np.expm1(centroids['log_monetary']).round(0)
centroids['n'] = rfm['cluster'].value_counts().sort_index().values
centroids['churn_pct'] = (
    rfm.merge(mf[['seller_id','is_churned']], on='seller_id')
    .groupby('cluster')['is_churned'].mean().sort_index().values
) * 100

print('Cluster centroids (raw scale):')
display_cols = ['recency_days','log_frequency_exp','log_monetary_exp','n','churn_pct']
print(centroids[display_cols].rename(columns={
    'recency_days': 'recency_days (median)',
    'log_frequency_exp': 'approx_order_count',
    'log_monetary_exp': 'approx_gmv',
    'n': 'size',
    'churn_pct': 'churned_%'
}).round(1).to_string())

Cluster centroids (raw scale):
   recency_days (median)  approx_order_count  approx_gmv  size  churned_%
0                   38.5                76.0      9688.0   645       13.2
1                   75.0                 2.0       153.0   748       33.2
2                   66.5                10.0      1434.0   999       27.4
3                  417.9                 3.0       251.0   568      100.0


In [7]:
# Label assignment based on RFM centroids:
#
#  Cluster 0: recency=12d,  freq=65,  GMV=high,  churn=13% → Champions
#             Most recent, highest volume, highest GMV. The platform's backbone.
#
#  Cluster 1: recency=44d,  freq=2,   GMV=low,   churn=33% → At Risk
#             Low frequency, moderate recency. Started selling but never scaled.
#             At risk of disengaging before they find product-market fit.
#
#  Cluster 2: recency=28d,  freq=10,  GMV=mid,   churn=27% → Rising Stars
#             Active and recent with moderate volume. Growth potential if supported.
#
#  Cluster 3: recency=406d, freq=2,   GMV=low,   churn=100% → Dormant
#             No orders in the last year. Effectively inactive.

CLUSTER_TO_SEGMENT = {
    0: 'Champions',
    1: 'At Risk',
    2: 'Rising Stars',
    3: 'Dormant',
}

rfm['segment'] = rfm['cluster'].map(CLUSTER_TO_SEGMENT)

# Merge segment labels back into the full merchant feature table
merchants = mf.merge(rfm[['seller_id', 'cluster', 'segment', 'log_frequency', 'log_monetary']], on='seller_id', how='left')
merchants = merchants.merge(mt, on='seller_id', how='left')

seg_counts = merchants['segment'].value_counts().reindex(SEGMENT_ORDER)
print('Segment distribution:')
for seg, n in seg_counts.items():
    print(f'  {seg:15s}: {n:>5} sellers ({n/len(merchants):.1%})')

Segment distribution:
  Champions      :   645 sellers (21.8%)
  Rising Stars   :   999 sellers (33.8%)
  At Risk        :   748 sellers (25.3%)
  Dormant        :   568 sellers (19.2%)


---
## 5. Segment Profile Table

One row per segment. Metrics: size, GMV share, median review score, median late delivery rate, median shipping days, top 3 categories, and share of merchants with a declining order trajectory.

In [8]:
total_gmv = merchants['total_gmv'].sum()

profile_rows = []
for seg in SEGMENT_ORDER:
    g = merchants[merchants['segment'] == seg]
    g_traj = g.dropna(subset=['is_declining'])

    top_cats = (
        g['top_category'].value_counts()
        .head(3).index.tolist()
    )

    profile_rows.append({
        'Segment':             seg,
        'Count':               len(g),
        'Share of Sellers':    f"{len(g)/len(merchants):.1%}",
        'GMV Share':           f"{g['total_gmv'].sum()/total_gmv:.1%}",
        'Median GMV (R$)':     f"{g['total_gmv'].median():,.0f}",
        'Median Orders':       f"{g['order_count'].median():.0f}",
        'Median Recency (d)':  f"{g['recency_days'].median():.0f}",
        'Median Review':       f"{g['avg_review_score'].median():.2f}",
        'Median Late Rate':    f"{g['pct_orders_late'].median():.1%}",
        'Median Ship Days':    f"{g['avg_shipping_days'].median():.1f}",
        '% Declining Traj':    f"{g_traj['is_declining'].mean()*100:.1f}%" if len(g_traj) > 0 else 'N/A',
        'Top Categories':      ', '.join(top_cats),
    })

profile_df = pd.DataFrame(profile_rows).set_index('Segment')
print('=== Segment Profile Table ===')
print(profile_df.T.to_string())

=== Segment Profile Table ===
Segment                                             Champions                               Rising Stars                          At Risk                                         Dormant
Count                                                     645                                        999                              748                                             568
Share of Sellers                                        21.8%                                      33.8%                            25.3%                                           19.2%
GMV Share                                               80.9%                                      15.0%                             1.2%                                            2.9%
Median GMV (R$)                                         8,393                                      1,349                              170                                             250
Median Orders                           

---
## 6. Visualizations — 2×2 Plotly Grid

In [9]:
fig_grid = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        '(a) Segment Size — Number of Sellers',
        '(b) GMV Concentration by Segment',
        '(c) Late Delivery Rate Distribution by Segment',
        '(d) Average Review Score Distribution by Segment',
    ),
    specs=[
        [{'type': 'bar'},    {'type': 'pie'}],
        [{'type': 'box'},    {'type': 'box'}],
    ],
    vertical_spacing=0.16,
    horizontal_spacing=0.10
)

seg_colors_ordered = [SEGMENT_COLORS[s] for s in SEGMENT_ORDER]

# ── (a) Bar: segment sizes ──────────────────────────────────────────────────
counts = merchants['segment'].value_counts().reindex(SEGMENT_ORDER)
fig_grid.add_trace(
    go.Bar(
        x=SEGMENT_ORDER,
        y=counts.values,
        marker_color=seg_colors_ordered,
        text=[f'{n:,}' for n in counts.values],
        textposition='outside',
        hovertemplate='%{x}<br>Sellers: %{y:,}<extra></extra>',
        showlegend=False,
    ),
    row=1, col=1
)

# ── (b) Donut: GMV share ────────────────────────────────────────────────────
gmv_by_seg = merchants.groupby('segment')['total_gmv'].sum().reindex(SEGMENT_ORDER)
fig_grid.add_trace(
    go.Pie(
        labels=SEGMENT_ORDER,
        values=gmv_by_seg.values,
        marker_colors=seg_colors_ordered,
        hole=0.45,
        textinfo='label+percent',
        textposition='outside',
        hovertemplate='%{label}<br>GMV: R$%{value:,.0f}<br>Share: %{percent}<extra></extra>',
        showlegend=False,
    ),
    row=1, col=2
)

# ── (c) Box: pct_orders_late by segment ────────────────────────────────────
for seg in SEGMENT_ORDER:
    seg_data = merchants[merchants['segment'] == seg]['pct_orders_late'].dropna() * 100
    fig_grid.add_trace(
        go.Box(
            y=seg_data,
            name=seg,
            marker_color=SEGMENT_COLORS[seg],
            boxmean='sd',
            hovertemplate=f'{seg}<br>Late rate: %{{y:.1f}}%<extra></extra>',
            showlegend=False,
        ),
        row=2, col=1
    )

# ── (d) Box: avg_review_score by segment ───────────────────────────────────
for seg in SEGMENT_ORDER:
    seg_data = merchants[merchants['segment'] == seg]['avg_review_score'].dropna()
    fig_grid.add_trace(
        go.Box(
            y=seg_data,
            name=seg,
            marker_color=SEGMENT_COLORS[seg],
            boxmean='sd',
            hovertemplate=f'{seg}<br>Review score: %{{y:.2f}}<extra></extra>',
            showlegend=False,
        ),
        row=2, col=2
    )

# ── Layout ──────────────────────────────────────────────────────────────────
fig_grid.update_yaxes(title_text='Number of Sellers', row=1, col=1, gridcolor='#eee')
fig_grid.update_yaxes(title_text='Late Delivery Rate (%)', row=2, col=1, gridcolor='#eee')
fig_grid.update_yaxes(title_text='Average Review Score (1–5)', row=2, col=2, gridcolor='#eee', range=[1, 5.5])
fig_grid.update_xaxes(showgrid=False, row=1, col=1)
fig_grid.update_xaxes(showgrid=False, row=2, col=1)
fig_grid.update_xaxes(showgrid=False, row=2, col=2)

fig_grid.update_layout(
    title=dict(
        text='Merchant Segmentation: RFM K-Means (k=4) — Size, GMV, Delivery Reliability, and Review Quality',
        font_size=15
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    height=800, margin=dict(t=100, b=60)
)

fig_grid.show()
print("Takeaway: Champions represent 22% of sellers but 81% of total GMV; "
      "their late delivery rate is meaningfully higher than the smaller segments, "
      "making fulfillment the highest-leverage intervention even in the platform's healthiest tier.")

Takeaway: Champions represent 22% of sellers but 81% of total GMV; their late delivery rate is meaningfully higher than the smaller segments, making fulfillment the highest-leverage intervention even in the platform's healthiest tier.


**Takeaway:** Champions make up just 22% of sellers but account for 81% of total GMV, so platform revenue is heavily concentrated in a small group. Champions also have the highest late delivery rates among active segments, which makes fulfillment reliability a high-priority area to dig into further.

---
## 7. Supplementary: Declining Trajectory Rate by Segment

Cross-reference segmentation with the trajectory analysis from Part 2. What share of each segment's merchants are on a declining order volume trajectory?

In [10]:
# Only merchants present in both time periods have a trajectory
traj_by_seg = (
    merchants
    .dropna(subset=['is_declining'])
    .groupby('segment')['is_declining']
    .agg(['mean', 'count'])
    .reindex(SEGMENT_ORDER)
    .reset_index()
)
traj_by_seg.columns = ['segment', 'pct_declining', 'n_with_traj']
traj_by_seg['pct_declining_pct'] = traj_by_seg['pct_declining'] * 100

fig_traj = go.Figure(
    go.Bar(
        x=traj_by_seg['segment'],
        y=traj_by_seg['pct_declining_pct'],
        marker_color=[SEGMENT_COLORS[s] for s in traj_by_seg['segment']],
        text=[f"{v:.1f}%<br>(n={n})" for v, n in zip(traj_by_seg['pct_declining_pct'], traj_by_seg['n_with_traj'])],
        textposition='outside',
        hovertemplate='%{x}<br>Declining: %{y:.1f}%<extra></extra>',
    )
)

fig_traj.update_layout(
    title=dict(
        text='Share of Merchants on Declining Order Volume Trajectory, by Segment<br>'
             '<sup>Among merchants active in both H1 and H2; Dormant merchants are excluded (no H2 activity by definition)</sup>',
        font_size=15
    ),
    xaxis_title='Segment',
    yaxis_title='% of Segment with Declining Trajectory',
    yaxis_range=[0, 70],
    plot_bgcolor='white', paper_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(gridcolor='#eee'),
    height=420, margin=dict(t=100)
)

fig_traj.show()
print('Declining trajectory rates by segment:')
for _, row in traj_by_seg.iterrows():
    print(f"  {row['segment']:15s}: {row['pct_declining_pct']:.1f}% declining (n={row['n_with_traj']} with trajectory data)")

print("\nTakeaway: Over half of Dormant merchants with trajectory data were already declining before "
      "going inactive — they were at-risk sellers who were never caught in time.")

Declining trajectory rates by segment:
  Champions      : 15.4% declining (n=434 with trajectory data)
  Rising Stars   : 29.3% declining (n=283 with trajectory data)
  At Risk        : 30.0% declining (n=30 with trajectory data)
  Dormant        : 52.8% declining (n=89 with trajectory data)

Takeaway: Over half of Dormant merchants with trajectory data were already declining before going inactive — they were at-risk sellers who were never caught in time.


**Takeaway:** The Dormant segment has the highest historical declining rate (53%), which suggests that dormancy often follows an unaddressed At Risk trajectory. Champions have the lowest declining rate (15%), reflecting their general operational stability. Rising Stars and At Risk both hover around 29-30% declining, which points to early intervention being valuable for both groups.

---
## 8. Save Segmented Merchant Table

In [11]:
out_path = PROCESSED / 'merchant_segments.csv'
merchants.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Shape: {merchants.shape}')
print(f'\nSegment column value counts:')
print(merchants['segment'].value_counts().reindex(SEGMENT_ORDER))
print(f'\nNull segments (sellers without RFM data): {merchants["segment"].isna().sum()}')

Saved: ../data/processed/merchant_segments.csv
Shape: (2960, 31)

Segment column value counts:
segment
Champions       645
Rising Stars    999
At Risk         748
Dormant         568
Name: count, dtype: int64

Null segments (sellers without RFM data): 0


---
## 9. Segment Summaries

Profile, key risk/opportunity, and product recommendation for each segment. Recommendation text is left for manual completion — the numbers are computed above.

---

In [12]:
# Print computed profile numbers to inform the written summaries below
for seg in SEGMENT_ORDER:
    g = merchants[merchants['segment'] == seg]
    g_traj = g.dropna(subset=['is_declining'])
    gmv_share = g['total_gmv'].sum() / total_gmv * 100
    dec_pct   = g_traj['is_declining'].mean() * 100 if len(g_traj) > 0 else float('nan')
    top_cats  = g['top_category'].value_counts().head(3).index.tolist()

    print(f"{'='*60}")
    print(f"{seg} — {len(g):,} merchants ({len(g)/len(merchants):.1%}), {gmv_share:.1f}% of GMV")
    print(f"  Median GMV:          R${g['total_gmv'].median():,.0f}")
    print(f"  Median orders:       {g['order_count'].median():.0f}")
    print(f"  Median recency:      {g['recency_days'].median():.0f} days")
    print(f"  Median review score: {g['avg_review_score'].median():.2f}")
    print(f"  Median late rate:    {g['pct_orders_late'].median():.1%}")
    print(f"  Median ship days:    {g['avg_shipping_days'].median():.1f}")
    print(f"  Declining trajectory:{dec_pct:.1f}% (of those with trajectory data)")
    print(f"  Top 3 categories:    {', '.join(top_cats)}")
    print()

Champions — 645 merchants (21.8%), 80.9% of GMV
  Median GMV:          R$8,393
  Median orders:       65
  Median recency:      12 days
  Median review score: 4.20
  Median late rate:    6.9%
  Median ship days:    12.3
  Declining trajectory:15.4% (of those with trajectory data)
  Top 3 categories:    health_beauty, sports_leisure, housewares

Rising Stars — 999 merchants (33.8%), 15.0% of GMV
  Median GMV:          R$1,349
  Median orders:       10
  Median recency:      28 days
  Median review score: 4.29
  Median late rate:    5.0%
  Median ship days:    11.1
  Declining trajectory:29.3% (of those with trajectory data)
  Top 3 categories:    sports_leisure, health_beauty, housewares

At Risk — 748 merchants (25.3%), 1.2% of GMV
  Median GMV:          R$170
  Median orders:       2
  Median recency:      44 days
  Median review score: 4.67
  Median late rate:    0.0%
  Median ship days:    8.7
  Declining trajectory:30.0% (of those with trajectory data)
  Top 3 categories:    health

---

### Champions (645 merchants, 80.9% of GMV)

**Profile:** 
Champions are the platform's most active and highest-value sellers. With a median of 65 orders and R$7,400 in GMV, they account for 81% of total platform revenue despite being only 22% of the seller base. They are recently active (median 12-day recency) and concentrated in health/beauty, sports/leisure, and housewares. Their late delivery rate of 6.9% (median) is the highest of any active segment, which stands out given how much of total platform revenue flows through this group.

**Key risk/opportunity:** 
Champions are heavily concentrated in platform GMV, so fulfillment failures in this segment ripple through the buyer experience in a significant way. A single Champion seller going dormant removes more GMV than 10 At Risk sellers combined, which highlights how much leverage there is in keeping this group healthy and engaged.

**Product recommendation:** 
* Champions aren't declining in large numbers (15.4%), but their 6.9% late rate is the highest of any active segment. At their volume, that means more late orders in absolute terms than any other group.
* They have figured out growth. Their bottleneck is keeping up with it operationally.
* A fulfillment performance dashboard showing their late rate trend over time, benchmarked against their category median, could help them spot problems before reviews drop.
* I'd also want to understand something I can't see in this data: are Champions with high late rates using the platform's shipping tools, or managing logistics independently? That would change whether the right intervention is better tooling or better defaults.

---

### Rising Stars (999 merchants, 15.0% of GMV)

**Profile:** Rising Stars are active, recently engaged sellers with moderate order volume (median 10 orders, R$720 GMV). They are the largest segment by count (34% of sellers) and represent the platform's growth pipeline, meaning merchants who have established a presence but have not yet reached Champion-level volume. Their review scores are solid (4.29 median) and their late delivery rate is low (5.0%), which suggests they are operationally capable at small scale.

**Key risk/opportunity:** 29% of Rising Stars with trajectory data show declining order volume, which means nearly one in three is stalling rather than growing. They currently contribute 15% of platform revenue, but this group likely holds the most future Champions if the platform can help them build momentum.

**Product recommendation:** 

* The alert I'd test: a weekly notification showing order volume trend vs. similar merchants in their category, paired with one specific suggestion (e.g., "merchants in health_beauty who added 3+ new SKUs this month saw 20% higher order volume").
* I'm honestly not sure whether the decline is supply-side (merchants pulling back) or demand-side (fewer buyers finding them). The data can't distinguish that. But either way, making the trend visible to the merchant is a low-cost first step.
* This is where I'd focus first if I had to pick one segment to build for, because the combination of decent metrics and declining volume means there's still time to intervene.

---

### At Risk (748 merchants, 1.2% of GMV)

**Profile:** At Risk merchants have low order volume (median 2 orders), moderate recency (44 days), and minimal GMV contribution (1.2% of total). Despite their small size, they have the highest review scores of any segment (4.67 median), which suggests that buyers who do purchase are genuinely satisfied. They likely represent merchants who joined the platform but struggled to gain traction. Their 0% median late delivery rate also shows that when they do fulfill, they do it reliably.

**Key risk/opportunity:** 30% of At Risk merchants with trajectory data are declining, and their high review scores suggest buyer relationships are still intact. The main barrier appears to be volume. If the platform can help these merchants acquire their next 10 orders, a subset would likely convert to Rising Stars.

**Product recommendation:** 

* These merchants likely never got enough orders to build momentum. Their buyers are happy (4.67 reviews), which suggests the product and service are fine.
* This looks like an onboarding and discoverability problem. The platform could test a "first 30 days" program: boosted visibility in search results or category pages for new merchants until they hit a threshold (maybe 10 orders: the point where merchant metrics stabilize enough to measure (our regression required it), and it's the median for Rising Stars, the next segment up). 
* I want to be careful here because I'm working with a dataset that doesn't include marketplace search or placement data. The recommendation is directional, not definitive. If I were at Shopify, I'd want to cross-reference this with data on merchant storefront views and conversion rates before investing in a solution.
* The question I'd ask the team: what does the platform's current onboarding flow look like, and at what point do most At Risk merchants stop engaging?

---

### Dormant (568 merchants, 2.9% of GMV)

**Profile:** Dormant merchants have not placed an order in over a year (median 406-day recency) and are 100% classified as churned under the proxy definition (no orders in the final 90 days of the dataset). They represent 19% of sellers but only 2.9% of GMV. Of the Dormant sellers who were active in both dataset halves, 53% had already been on a declining trajectory before going inactive, suggesting they were At Risk merchants the platform did not catch in time.

**Key risk/opportunity:** The Dormant segment is most useful as a diagnostic lens: it shows where the platform experience broke down before merchants disengaged entirely. Re-engaging Dormant sellers tends to be high-cost and low-yield, so the higher-value opportunity is probably in preventing At Risk and Rising Star merchants from ever reaching this state.

**Product recommendation:**
* I don't think a reactivation campaign is the right investment here. These merchants have been gone for 400+ days, and the typical ROI on win-back campaigns for long-inactive users is low.
* The real value of this segment is as a training signal. If 53% were visibly declining before they left, the platform could use those early trajectory patterns to flag current Rising Stars and At Risk merchants who are on the same path.
* Something like: "This merchant's order volume pattern over the last 60 days matches the trajectory of merchants who went dormant within 6 months." That kind of early warning, surfaced to a merchant success team or built into an automated alert, would be more impactful than trying to bring back merchants who already left.
* This is the segment where I have the least confidence in a specific product recommendation, and I think that's the honest answer. The intervention for Dormant merchants is really about not creating more of them.

---

### The Merchant Lifecycle Funnel

Reading across these four segments, the pattern is a leaky funnel:

At Risk merchants join the platform but never reach enough volume to build momentum (748 merchants, only 2 median orders). The ones who do break through become Rising Stars, but 29% of those stall and start declining despite healthy reviews. The Rising Stars who sustain growth eventually become Champions, but at that scale, fulfillment strain shows up as the highest late delivery rate on the platform. And Dormant merchants are what happens when the platform misses the signals at each earlier stage: 53% were already declining before they went inactive.

The interventions map to stages of this funnel, not to independent problems:

1. **Onboarding and discoverability** to move At Risk merchants past the 10-order traction threshold.
2. **Proactive fulfillment alerts** to catch Rising Stars before silent decline becomes visible decline. This is the highest-leverage intervention and the one I designed an experiment for (see [experiment design](../docs/experiment_design.md)).
3. **Operational benchmarking at scale** to help Champions manage the fulfillment strain that comes with volume.
4. **Early warning signals** built from Dormant trajectory patterns, applied to merchants still in the first three stages.

If I had to recommend one place to start, it would be Rising Stars. They're the largest segment (999 merchants), they're still active enough to respond, and they sit at the inflection point between growth and decline. The experiment I designed in Part 5 targets exactly this group. But the longer-term product strategy is the full funnel: reduce leakage at each transition, starting where the data says the biggest gap is.